# **Chapter 29: Magpie**

## **Section 1: 事前学習モデルの読み込み**

In [57]:
# added top-p and top-k filtering in generate function
# set vocab_size in config.py
# MHA with KV cache + RoPE + PyTorch SDPA.
# This traditional implementation is easier to understand, and still efficient in practice.
# GQA and MLA is a great way for long-text inference with reduced KV cache size,
# but both comes with slight loss increase and no efficiency merits during training phase.
# KV cache does not help training speed. Codebase will be simpler without it.
# KV cache supports multi-turn continuation by RoPE with position offset.
# No Dropout. Dataset is large enough and regularization is not necessary.

import torch
import torch.nn as nn
import torch.nn.functional as F

class TokenEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.embedding_dim)
        # keep embedding in default dtype (autocast will handle bf16 when enabled)

    def forward(self, input_indices):
        return self.token_embedding_table(input_indices)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048, rope_theta=1e6):
        super().__init__()

        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2) / dim))
        position_index = torch.arange(max_seq_len)
        frequency_matrix = torch.einsum('i,j->ij', position_index, inv_freq)

        cosine = torch.cos(frequency_matrix)[None, None, :, :]
        sine = torch.sin(frequency_matrix)[None, None, :, :]

        self.register_buffer("cos_cached", cosine, persistent=False)
        self.register_buffer("sin_cached", sine, persistent=False)

    def apply_rotary_emb(self, x, position_offset=0):
        sequence_length = x.size(2)

        cosine = self.cos_cached[:, :, position_offset:position_offset + sequence_length, :]
        sine = self.sin_cached[:, :, position_offset:position_offset + sequence_length, :]

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]

        rotated_even = x_even * cosine - x_odd * sine
        rotated_odd = x_odd * cosine + x_even * sine

        rotated = torch.empty_like(x)
        rotated[..., 0::2] = rotated_even
        rotated[..., 1::2] = rotated_odd

        return rotated

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_heads = config.num_attention_heads
        self.embed_dim = config.embedding_dim
        self.head_dim = self.embed_dim // self.num_heads

        # QKV projection
        self.query_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.key_fc   = nn.Linear(self.embed_dim, self.embed_dim, bias=False)
        self.value_fc = nn.Linear(self.embed_dim, self.embed_dim, bias=False)

        # Rotary Positional Embedding (RoPE)
        self.rotary_emb = RotaryEmbedding(
            dim=self.head_dim,
            max_seq_len=config.max_sequence_length,
            rope_theta=config.rope_theta
        )

        self.output_projection = nn.Linear(self.embed_dim, self.embed_dim)

        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(
                config.max_sequence_length,
                config.max_sequence_length,
                dtype=torch.bool
            )),
            persistent=False
        )

        # KV cache
        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)
        self.current_pos = 0

    # --------------------------------------------------
    # router
    # --------------------------------------------------
    def forward(self, x, use_cache=False):
        input_len = x.size(1)
        if use_cache is False:
            return self.forward_no_cache(x)
        elif use_cache is True and input_len > 1:
            return self.forward_prefill(x)
        elif use_cache is True and input_len == 1: # Hi scenario also starts with T==1
            return self.forward_cached_decoding(x)
        else:
            raise RuntimeError("Unexpected condition in MultiHeadAttention forward")

    # --------------------------------------------------
    # (1) no cache : training 
    # --------------------------------------------------
    def forward_no_cache(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # RoPE : offset = 0
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=0)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=0)

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=True
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (2) prefill : initialize KV cache
    # --------------------------------------------------
    def forward_prefill(self, x):
        B, T, C = x.shape

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # init cache
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        # RoPE : offset = current_pos (supports multi-turn continuation)
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        # prevent overflow
        if self.current_pos + T > self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        self.cache_k[:, :, self.current_pos:self.current_pos + T, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + T, :] = V

        K = self.cache_k[:, :, :self.current_pos + T, :]
        V = self.cache_v[:, :, :self.current_pos + T, :]

        attn_mask = self.causal_mask[
            self.current_pos : self.current_pos + T,
            : self.current_pos + T
        ]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            is_causal=False
        )

        self.current_pos += T

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    # --------------------------------------------------
    # (3) decode : cached decoding (1 token)
    # --------------------------------------------------
    def forward_cached_decoding(self, x):
        B, T, C = x.shape
        assert T == 1, "cached decoding expects T==1"

        Q = self.query_fc(x)
        K = self.key_fc(x)
        V = self.value_fc(x)

        Q = Q.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, 1, self.num_heads, self.head_dim).transpose(1, 2)

        # This is not usually needed since prefill should have initialized the cache.
        # Just in case for "Hi" scenario, which starts with single token input.
        if self.cache_k is None:
            self.cache_k = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=K.dtype
            )
            self.cache_v = torch.zeros(
                B, self.num_heads, self.config.max_sequence_length, self.head_dim,
                device=x.device, dtype=V.dtype
            )
            self.current_pos = 0

        if self.current_pos + 1 >= self.config.max_sequence_length:
            raise RuntimeError("KV cache exceeded max_sequence_length")

        # RoPE : offset = current_pos
        Q = self.rotary_emb.apply_rotary_emb(Q, position_offset=self.current_pos)
        K = self.rotary_emb.apply_rotary_emb(K, position_offset=self.current_pos)

        self.cache_k[:, :, self.current_pos:self.current_pos + 1, :] = K
        self.cache_v[:, :, self.current_pos:self.current_pos + 1, :] = V

        K = self.cache_k[:, :, :self.current_pos + 1, :]
        V = self.cache_v[:, :, :self.current_pos + 1, :]

        out = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=None,
            is_causal=False
        )

        self.current_pos += 1

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.output_projection(out)
        return out

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.current_pos = 0



class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()    
        self.net = nn.Sequential(
            nn.Linear(config.embedding_dim, config.hidden_dim, bias=False),
            nn.ReLU(),
            nn.Linear(config.hidden_dim, config.embedding_dim, bias=False),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(config.embedding_dim)
        self.layer_norm2 = nn.LayerNorm(config.embedding_dim)
        self.multihead_attention = MultiHeadAttention(config=config)
        self.feed_forward = FeedForward(config=config)


    def forward(self, input_tensor, use_cache=False):
        normed_input = self.layer_norm1(input_tensor)
        attention_output = self.multihead_attention(normed_input, use_cache=use_cache)
        residual_attention = attention_output + input_tensor
        normed_attention = self.layer_norm2(residual_attention)
        feedforward_output = self.feed_forward(normed_attention)
        final_output = feedforward_output + residual_attention
        return final_output


class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(config.embedding_dim)
        self.vocab_projection = nn.Linear(config.embedding_dim, config.vocab_size, bias=False)

    def forward(self, transformer_block_output):
        x = transformer_block_output
        normalized_output = self.output_norm(x)
        vocab_logits = self.vocab_projection(normalized_output)
        return vocab_logits


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_layer = TokenEmbedding(config=config)
        self.blocks = nn.ModuleList([TransformerBlock(config=config) for _ in range(config.layer_count)])
        self.vocab_projection = VocabularyLogits(config=config)
        self.criterion = nn.CrossEntropyLoss()


    def forward(self, input_indices, target_indices, use_cache=False):
        token_embeddings = self.token_embedding_layer.forward(input_indices)

        x = token_embeddings
        for block in self.blocks:
            x = block(x, use_cache=use_cache)
        logits = self.vocab_projection(x)

        if target_indices is None:
            return logits, None

        batch_size, token_len, vocab_size = logits.shape
        logits_flat = logits.view(batch_size * token_len, vocab_size)
        targets_flat = target_indices.view(batch_size * token_len)
        loss = self.criterion(logits_flat, targets_flat)
        return logits, loss


    def generate(self,
        input_indices,
        max_new_tokens,
        temperature=1.0,
        use_cache=True,
        reset_cache=False,
        top_k=None,      # ### NEW ###
        top_p=None,      # ### NEW ###
    ):
        self.eval()

        if reset_cache:
            for block in self.blocks:
                block.multihead_attention.reset_cache()

        next_token = None

        for i in range(max_new_tokens):
            if use_cache:
                if i == 0:
                    logits, _ = self.forward(input_indices, None, use_cache=True)
                else:
                    logits, _ = self.forward(next_token, None, use_cache=True)
            else:
                logits, _ = self.forward(input_indices, None, use_cache=False)

            """ DELETE
            last_logits = logits[:, -1, :] / temperature
            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            """

            ### NEW ###
            last_logits = logits[:, -1, :] / temperature

            if top_k is not None:
                top_k = min(top_k, last_logits.size(-1))
                values, _ = torch.topk(last_logits, top_k)
                min_value = values[:, -1].unsqueeze(-1)
                last_logits = torch.where(
                    last_logits < min_value,
                    torch.full_like(last_logits, float("-inf")),
                    last_logits,
                )

            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(last_logits, descending=True)
                sorted_probs = F.softmax(sorted_logits, dim=-1)
                cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

                sorted_mask = cumulative_probs > top_p
                sorted_mask[..., 1:] = sorted_mask[..., :-1].clone()
                sorted_mask[..., 0] = False

                sorted_logits = torch.where(
                    sorted_mask,
                    torch.full_like(sorted_logits, float("-inf")),
                    sorted_logits,
                )

                last_logits = torch.zeros_like(last_logits).scatter(
                    -1, sorted_indices, sorted_logits
                )

            probs = F.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            ### NEW ###

            yield int(next_token.item())
            input_indices = torch.cat((input_indices, next_token), dim=1)

In [58]:
class Config:
    embedding_dim: int = 2560
    hidden_dim: int = 10240
    num_attention_heads: int = 20
    layer_count: int = 30
    rope_theta: float = 1_000_000.0
    vocab_size: int = 50257
    max_sequence_length: int = 2048

In [59]:
config = Config()
model = GPT(config)
device = torch.device("cuda")

In [60]:
import random
RANDOM_SEED = 1337
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

In [61]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="HayatoHongo/AIkenSGTv1",
    filename="model_clean.safetensors",
)
print(model_path)

/root/.cache/huggingface/hub/models--HayatoHongo--AIkenSGTv1/snapshots/1463c8860b77b002303c2f643f29078f2aad80e5/model_clean.safetensors


In [62]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [63]:
from safetensors.torch import load_file
state_dict = load_file(model_path, device="cpu")
model.load_state_dict(state_dict)

<All keys matched successfully>

In [64]:
# ⚠️ Don't run this cell twice!
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
compiled_model = torch.compile(model)

In [65]:
compiled_model = compiled_model.to(device)
compiled_model.eval()

OptimizedModule(
  (_orig_mod): GPT(
    (token_embedding_layer): TokenEmbedding(
      (token_embedding_table): Embedding(50257, 2560)
    )
    (blocks): ModuleList(
      (0-29): 30 x TransformerBlock(
        (layer_norm1): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (layer_norm2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (multihead_attention): MultiHeadAttention(
          (query_fc): Linear(in_features=2560, out_features=2560, bias=False)
          (key_fc): Linear(in_features=2560, out_features=2560, bias=False)
          (value_fc): Linear(in_features=2560, out_features=2560, bias=False)
          (rotary_emb): RotaryEmbedding()
          (output_projection): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (feed_forward): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=2560, out_features=10240, bias=False)
            (1): ReLU()
            (2): Linear(in_features=10240, out_feature

In [66]:
prompt = "人工知能とは"
input_indices = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)

In [67]:
with torch.no_grad():
    generated_ids = list(compiled_model.generate(input_indices, max_new_tokens=50, temperature=0.8, top_k=50, top_p=0.9))
print(prompt + tokenizer.decode(generated_ids))

人工知能とは？
- 人工知能（AI）とは？
- 人工知能（AI）の今後



---

期待する形式

以下のデータセットと同じ形式

```plain
{"prompt": "What is the capital of Japan?", "response": "The capital of Japan is Tokyo."}
{"prompt": "Hello there!", "response": "Hello! How can I assist you today?"}
{"prompt": "What is the result of 2^3?", "response": "The result of 2^3 is 8."}
```

In [68]:
from huggingface_hub import hf_hub_download

hf_hub_download(
    repo_id="HayatoHongo/AIkenSGTv1",
    repo_type="model",
    filename="train_prompt_response.jsonl",
    local_dir="/content",
)

'/content/train_prompt_response.jsonl'

In [69]:
import json
tokenizer = tiktoken.get_encoding("gpt2")
config.input_sequence_length = 1024

jsonl_file = open("/content/train_prompt_response.jsonl", "r", encoding="utf-8")
output_jsonl_file = open("/content/sft_token_pairs.jsonl", "w", encoding="utf-8")

for json_line in jsonl_file:
    sample = json.loads(json_line)

    prompt_text = "<USER>" + sample["prompt"] + "<ASSISTANT>"
    response_text = sample["response"] + "<|endoftext|>"

    prompt_ids = tokenizer.encode(prompt_text)
    response_ids = tokenizer.encode(response_text, allowed_special="all")

    chunk = prompt_ids + response_ids
    input_ids = chunk[:-1]

    prompt_masked_ids = [-100] * len(prompt_ids)
    prompt_masked_chunk = prompt_masked_ids + response_ids
    target_ids = prompt_masked_chunk[1:]

    padding_length = config.input_sequence_length - len(input_ids)
    padded_input_ids = input_ids + [50256] * padding_length
    padded_target_ids = target_ids + [-100] * padding_length

    output_dict = {"padded_input_ids": padded_input_ids, "padded_target_ids": padded_target_ids,}
    output_json_line = json.dumps(output_dict)
    output_jsonl_file.write(output_json_line)
    output_jsonl_file.write("\n")

jsonl_file.close()
output_jsonl_file.close()

In [70]:
padded_input_ids_list = []
jsonl_file = open("/content/sft_token_pairs.jsonl", "r", encoding="utf-8")

for json_line in jsonl_file:
    sample = json.loads(json_line)
    padded_input_ids_list.append(sample["padded_input_ids"])

input_ids_tensor = torch.tensor(padded_input_ids_list, dtype=torch.long)
jsonl_file.close()

torch.save(input_ids_tensor, "/content/input_ids.pt")
print("input_ids:", input_ids_tensor.shape)

input_ids: torch.Size([7220, 1024])


In [71]:
from huggingface_hub import login
login()

In [72]:
# CPUメモリを解放するために、不要な変数を削除してガベージコレクションを実行します。
import gc
del padded_input_ids_list, input_ids_tensor
gc.collect()

9

In [73]:
padded_target_ids_list = []
jsonl_file = open("/content/sft_token_pairs.jsonl", "r", encoding="utf-8")

for json_line in jsonl_file:
    sample = json.loads(json_line)
    padded_target_ids_list.append(sample["padded_target_ids"])

target_ids_tensor = torch.tensor(padded_target_ids_list, dtype=torch.long)
jsonl_file.close()

torch.save(target_ids_tensor, "/content/target_ids.pt")
print("target_ids:", target_ids_tensor.shape)

target_ids: torch.Size([7220, 1024])


In [74]:
from huggingface_hub import upload_file
model_repo_id = "HayatoHongo/AIkenSGTv1"
model_path = "/content/target_ids.pt"
upload_file(repo_id=model_repo_id, repo_type="model", path_or_fileobj=model_path, path_in_repo=model_path)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /content/target_ids.pt      : 100%|##########| 59.1MB / 59.1MB            

CommitInfo(commit_url='https://huggingface.co/HayatoHongo/AIkenSGTv1/commit/cc4ecf8b202a68efd545fe609b59ae1e2c389e90', commit_message='Upload /content/target_ids.pt with huggingface_hub', commit_description='', oid='cc4ecf8b202a68efd545fe609b59ae1e2c389e90', pr_url=None, repo_url=RepoUrl('https://huggingface.co/HayatoHongo/AIkenSGTv1', endpoint='https://huggingface.co', repo_type='model', repo_id='HayatoHongo/AIkenSGTv1'), pr_revision=None, pr_num=None)

In [75]:
# CPUメモリを解放するために、不要な変数を削除してガベージコレクションを実行します。
import gc
del padded_target_ids_list, target_ids_tensor
gc.collect()

118

In [76]:
class DataLoader:
    def __init__(self, input_ids_tensor_path, target_ids_tensor_path, config):
        self.config = config
        self.input_ids = torch.load(input_ids_tensor_path, mmap=True)
        self.target_ids = torch.load(target_ids_tensor_path, mmap=True)
        self.data_size = len(self.input_ids)
        self.current_index = 0

    def get_batch(self):
        start_index = self.current_index
        end_index = start_index + self.config.batch_size

        if end_index > self.data_size:
            start_index = 0
            end_index = self.config.batch_size

        input_batch = self.input_ids[start_index:end_index]
        target_batch = self.target_ids[start_index:end_index]
        self.current_index = end_index

        input_batch = input_batch.to(self.config.device_type)
        target_batch = target_batch.to(self.config.device_type)

        return input_batch, target_batch

In [77]:
data_loader = DataLoader(
    input_ids_tensor_path="/content/input_ids.pt",
    target_ids_tensor_path="/content/target_ids.pt",
    config=config,
)

In [78]:
def get_learning_rate(current_step, config):
    max_learning_rate = config.max_learning_rate
    min_learning_rate = config.min_learning_rate
    warmup_steps = config.warmup_steps
    total_steps = config.total_steps

    if current_step < warmup_steps:
        # --- Linear Warmup ---
        warmup_progress_ratio = current_step / warmup_steps
        learning_rate = max_learning_rate * warmup_progress_ratio

    else:
        # --- Linear Decay ---
        decay_total_steps = total_steps - warmup_steps
        decay_step_index = current_step - warmup_steps

        decay_progress_ratio = decay_step_index / decay_total_steps
        learning_rate_range = max_learning_rate - min_learning_rate
        learning_rate = max_learning_rate - learning_rate_range * decay_progress_ratio

    return learning_rate


In [79]:
import os

def save_checkpoint(model, optimizer, checkpoint_dir, step):
    os.makedirs(checkpoint_dir, exist_ok=True)

    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"checkpoint_{step:06d}.pt",
    )

    checkpoint_data = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }

    torch.save(checkpoint_data, checkpoint_path)
    print(f"[INFO] Successfully saved checkpoint at step {step:06d}")


def load_checkpoint(model, optimizer, checkpoint_dir, step, device):
    checkpoint_path = os.path.join(
        checkpoint_dir,
        f"checkpoint_{step:06d}.pt",
    )

    checkpoint_data = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint_data["model_state_dict"])
    optimizer.load_state_dict(checkpoint_data["optimizer_state_dict"])
    del checkpoint_data
    gc.collect()
    print(f"[INFO] Resume completed at step {step}")

In [80]:
import time

from huggingface_hub import hf_hub_download
from huggingface_hub import create_repo
from huggingface_hub import upload_file


class Trainer:
    def __init__(self, model, optimizer, data_loader, config, checkpoint_dir, repo_id, start_step=0):
        self.model = model
        self.optimizer = optimizer
        self.data_loader = data_loader
        self.config = config
        self.checkpoint_dir = checkpoint_dir
        self.repo_id = repo_id
        self.start_step = start_step

        self.steps = []
        self.learning_rates = []
        self.train_losses = []
        self.tokens_per_second_list = []
        self.total_seen_tokens_list = []
        self.total_train_time_list = []

    def train_step(self):
        accumulation_steps = self.config.gradient_accumulation_steps
        self.optimizer.zero_grad()
        accumulated_loss = 0.0

        for _ in range(accumulation_steps):
            input_batch, target_batch = self.data_loader.get_batch()

            with torch.autocast(device_type=self.config.device_type, dtype=torch.bfloat16,):
                _, loss = self.model(input_batch, target_batch)

            (loss / accumulation_steps).backward() # 本体
            accumulated_loss += loss.item() # 表示用

        self.optimizer.step() # 勾配がここでリセットされる

        return accumulated_loss / accumulation_steps

    def train(self):
        if self.start_step > 0:
            checkpoint_filename = f"checkpoint_{self.start_step:06d}.pt"
            checkpoint_path = os.path.join(self.checkpoint_dir, checkpoint_filename)
            hf_hub_download(repo_id=self.repo_id, filename=checkpoint_path, local_dir="/content")
            load_checkpoint(
                self.model,
                self.optimizer,
                self.checkpoint_dir,
                self.start_step,
                self.config.device_type,
            )

        total_train_time = 0
        last_log_time = time.time()

        for step in range(self.start_step + 1, self.config.total_steps + 1):
            current_learning_rate = get_learning_rate(step, self.config)
            self.optimizer.param_groups[0]["lr"] = current_learning_rate

            train_loss = self.train_step()

            if step % self.config.checkpoint_save_frequency == 0:
                save_checkpoint(self.model, self.optimizer, self.checkpoint_dir, step)

                create_repo(repo_id=self.repo_id, private=False, exist_ok=True)
                checkpoint_filename = f"checkpoint_{step:06d}.pt"
                checkpoint_path = os.path.join(self.checkpoint_dir, checkpoint_filename)
                upload_file(repo_id=self.repo_id, path_or_fileobj=checkpoint_path, path_in_repo=checkpoint_path)

            current_log_time = time.time()
            interval = current_log_time - last_log_time
            total_train_time += interval

            tokens_per_interval = (self.config.global_batch_size * self.config.input_sequence_length)
            tokens_per_second = tokens_per_interval / interval if interval > 0 else None
            total_seen_tokens = tokens_per_interval * step

            print(
                f"step {step:05d} | ",
                f"lr {current_learning_rate:.6e} | ",
                f"train loss {train_loss:.4f} | ",
                f"tok/s {int(tokens_per_second) if tokens_per_second is not None else 'None'} | ",
                f"tokens {total_seen_tokens:,} | ",
                f"time {total_train_time:.2f}s",
            )

            self.steps.append(step)
            self.learning_rates.append(current_learning_rate)
            self.train_losses.append(train_loss)
            self.tokens_per_second_list.append(tokens_per_second)
            self.total_seen_tokens_list.append(total_seen_tokens)
            self.total_train_time_list.append(total_train_time)

            last_log_time = current_log_time

In [81]:
# close to nanoGPT setting
optimizer = torch.optim.AdamW(
    model.parameters(),
    betas=(0.9, 0.95),
    weight_decay=0.0,
    fused=True,
)

In [82]:
compiled_model.train()

OptimizedModule(
  (_orig_mod): GPT(
    (token_embedding_layer): TokenEmbedding(
      (token_embedding_table): Embedding(50257, 2560)
    )
    (blocks): ModuleList(
      (0-29): 30 x TransformerBlock(
        (layer_norm1): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (layer_norm2): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (multihead_attention): MultiHeadAttention(
          (query_fc): Linear(in_features=2560, out_features=2560, bias=False)
          (key_fc): Linear(in_features=2560, out_features=2560, bias=False)
          (value_fc): Linear(in_features=2560, out_features=2560, bias=False)
          (rotary_emb): RotaryEmbedding()
          (output_projection): Linear(in_features=2560, out_features=2560, bias=True)
        )
        (feed_forward): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=2560, out_features=10240, bias=False)
            (1): ReLU()
            (2): Linear(in_features=10240, out_feature

In [83]:
from huggingface_hub import login
login()

In [84]:
config.batch_size = 1
config.global_batch_size = 32
config.gradient_accumulation_steps = config.global_batch_size // config.batch_size

In [85]:
config.max_learning_rate = 1e-4
config.min_learning_rate = 1e-5
config.warmup_steps = 50
config.checkpoint_save_frequency = 2_000
config.device_type = "cuda"

In [86]:
data_size = data_loader.data_size
num_epoch = 1
config.total_steps = data_size * num_epoch // (config.batch_size * config.gradient_accumulation_steps)
print(f"Total training steps: {config.total_steps}")

Total training steps: 225


In [87]:
model_repo_id = "HayatoHongo/AIkenSGTv1"

In [88]:
trainer = Trainer(
    model=compiled_model,
    optimizer=optimizer,
    data_loader=data_loader,
    config=config,
    checkpoint_dir="./checkpoints-prompt-mask-instruction-tuning-easy-jmmlu",
    repo_id = model_repo_id,
    start_step=0,
)

In [89]:
trainer.train()

step 00001 |  lr 2.000000e-06 |  train loss 7.1620 |  tok/s 389 |  tokens 32,768 |  time 84.04s
step 00002 |  lr 4.000000e-06 |  train loss 4.5169 |  tok/s 8041 |  tokens 65,536 |  time 88.12s
step 00003 |  lr 6.000000e-06 |  train loss 1.9720 |  tok/s 8236 |  tokens 98,304 |  time 92.10s
step 00004 |  lr 8.000000e-06 |  train loss 0.8252 |  tok/s 8226 |  tokens 131,072 |  time 96.08s
step 00005 |  lr 1.000000e-05 |  train loss 0.7824 |  tok/s 8227 |  tokens 163,840 |  time 100.06s
step 00006 |  lr 1.200000e-05 |  train loss 0.8074 |  tok/s 8228 |  tokens 196,608 |  time 104.04s
step 00007 |  lr 1.400000e-05 |  train loss 0.8184 |  tok/s 8231 |  tokens 229,376 |  time 108.03s
step 00008 |  lr 1.600000e-05 |  train loss 0.7352 |  tok/s 8223 |  tokens 262,144 |  time 112.01s
step 00009 |  lr 1.800000e-05 |  train loss 0.7109 |  tok/s 8228 |  tokens 294,912 |  time 115.99s
step 00010 |  lr 2.000000e-05 |  train loss 0.6671 |  tok/s 8260 |  tokens 327,680 |  time 119.96s
step 00011 |  lr 2

In [ ]:
# 学習が完了したら推論もやってみる
compiled_model.eval()
prompt = "<USER>次の問題に、A〜Dから1つ選んで答えてください。\n\n問題: 電子機器で使用される最も主要な電子回路基板の事をなんと言う？\nA. 掲示板\nB. パソコン\nC. マザーボード\nD. ハードディスク\n\n回答:<ASSISTANT>"
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(prompt, allowed_special="all") # テキストをIDにエンコード
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # IDのリストをテンソルの形式に変換する
encoded_tensor = encoded_tensor.unsqueeze(0)  # バッチ次元追加
encoded_tensor = encoded_tensor.to(config.device_type) # cuda(GPU)にencoded_tensorを転送する
generated_tensor = compiled_model.generate(encoded_tensor, max_new_tokens=64, temperature=0.5)
generated_ids = list(generated_tensor)
generated_text = tokenizer.decode(generated_ids)
print(generated_text)

AttributeError: 'generator' object has no attribute 'squeeze'

In [ ]:
compiled_model.eval()
prompt = "<USER>"
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(prompt, allowed_special="all") # テキストをIDにエンコード
encoded_tensor = torch.tensor(encoded, dtype=torch.long) # IDのリストをテンソルの形式に変換する
encoded_tensor = encoded_tensor.unsqueeze(0)  # バッチ次元追加
encoded_tensor = encoded_tensor.to(config.device_type) # cuda(GPU)にencoded_tensorを転送する
generated_tensor = compiled_model.generate(encoded_tensor, max_new_tokens=64, temperature=0.9) # 温度を高めに設定
generated_ids = generated_tensor.squeeze(0).tolist() # バッチ次元を削除してリストに変換する
generated_text = tokenizer.decode(generated_ids) # IDのリストをテキストにデコードする
print(generated_text)

In [ ]:
from safetensors.torch import save_file

# torch.compile wrapperを外し、元のモデルのstate_dictを保存する
model_to_save = getattr(compiled_model, "_orig_mod", compiled_model)
state_dict = model_to_save.state_dict()
assert not any(key.startswith("_orig_mod.") for key in state_dict)

model_basename = f"prompt_mask_instruction_tuned_model_epoch_{num_epoch}_lr_{config.max_learning_rate:.0e}_easy_jmmlu"
safetensors_path = model_basename + ".safetensors"
cpu_state_dict = {
    key: tensor.detach().to("cpu").contiguous()
    for key, tensor in state_dict.items()
}
save_file(cpu_state_dict, safetensors_path, metadata={"format": "pt"})
del cpu_state_dict


In [ ]:
upload_file(
    repo_id=model_repo_id,
    repo_type="model",
    path_or_fileobj=safetensors_path,
    path_in_repo=safetensors_path,
)


In [ ]:
# グラフ描画。
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 5))
plt.plot(trainer.steps, trainer.train_losses, label='Train Loss')

plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training and Validation Loss over Steps')
plt.legend()
plt.grid(True)
plt.show()